In [1]:
import json
import math
import os
from pathlib import Path
from pprint import pprint
import sys

path = "/home/lk3591/Documents/code/RawByteClf"
os.chdir(path)
if path not in sys.path:
	sys.path.insert(0, path)

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from tokenizers import models, pre_tokenizers, processors, Tokenizer, Regex
from transformers import PreTrainedTokenizer, PreTrainedTokenizerFast

from src.learn.tokenization import SPECIALS

Entered __file__='/home/lk3591/Documents/code/RawByteClf/src/__init__.py'
Entered __file__='/home/lk3591/Documents/code/RawByteClf/src/learn/__init__.py'
Entered __file__='/home/lk3591/Documents/code/RawByteClf/src/learn/tokenization.py'


In [7]:
SPECIALS_IDS = {k: i for i, k in enumerate(SPECIALS)}
SPECIALS, SPECIALS_IDS

(OrderedDict([('pad_token', '<pad>'),
              ('unk_token', '<unk>'),
              ('mask_token', '<msk>'),
              ('bos_token', '<bos>'),
              ('eos_token', '<eos>'),
              ('cls_token', '<cls>'),
              ('sep_token', '<sep>')]),
 {'pad_token': 0,
  'unk_token': 1,
  'mask_token': 2,
  'bos_token': 3,
  'eos_token': 4,
  'cls_token': 5,
  'sep_token': 6})

In [2]:
file = "/home/lk3591/Documents/datasets/BODMAS/binaries/00000afff2659baffcbb11b90aedecc1d46754599282c918aa7ea5a9b0d8d724.exe"
max_length = None
byte = open(file, "rb").read(max_length)
text = byte[0:max_length].decode("latin1")
len(byte), len(text)

(784660, 784660)

In [3]:
correct_ids = [b + len(SPECIALS) for b in byte]
byte[:8], text[0:8], [b for b in byte[:8]], correct_ids[:8]

(b'MZP\x00\x02\x00\x00\x00',
 'MZP\x00\x02\x00\x00\x00',
 [77, 90, 80, 0, 2, 0, 0, 0],
 [84, 97, 87, 7, 9, 7, 7, 7])

In [4]:
alphabet = [bytes([i]).decode("latin1") for i in range(256)]
vocab = {v: i for i, v in enumerate(SPECIALS.values())}
vocab.update({v: i for i, v in enumerate(alphabet, start=len(SPECIALS))})
vocab

{'<pad>': 0,
 '<unk>': 1,
 '<msk>': 2,
 '<bos>': 3,
 '<eos>': 4,
 '<cls>': 5,
 '<sep>': 6,
 '\x00': 7,
 '\x01': 8,
 '\x02': 9,
 '\x03': 10,
 '\x04': 11,
 '\x05': 12,
 '\x06': 13,
 '\x07': 14,
 '\x08': 15,
 '\t': 16,
 '\n': 17,
 '\x0b': 18,
 '\x0c': 19,
 '\r': 20,
 '\x0e': 21,
 '\x0f': 22,
 '\x10': 23,
 '\x11': 24,
 '\x12': 25,
 '\x13': 26,
 '\x14': 27,
 '\x15': 28,
 '\x16': 29,
 '\x17': 30,
 '\x18': 31,
 '\x19': 32,
 '\x1a': 33,
 '\x1b': 34,
 '\x1c': 35,
 '\x1d': 36,
 '\x1e': 37,
 '\x1f': 38,
 ' ': 39,
 '!': 40,
 '"': 41,
 '#': 42,
 '$': 43,
 '%': 44,
 '&': 45,
 "'": 46,
 '(': 47,
 ')': 48,
 '*': 49,
 '+': 50,
 ',': 51,
 '-': 52,
 '.': 53,
 '/': 54,
 '0': 55,
 '1': 56,
 '2': 57,
 '3': 58,
 '4': 59,
 '5': 60,
 '6': 61,
 '7': 62,
 '8': 63,
 '9': 64,
 ':': 65,
 ';': 66,
 '<': 67,
 '=': 68,
 '>': 69,
 '?': 70,
 '@': 71,
 'A': 72,
 'B': 73,
 'C': 74,
 'D': 75,
 'E': 76,
 'F': 77,
 'G': 78,
 'H': 79,
 'I': 80,
 'J': 81,
 'K': 82,
 'L': 83,
 'M': 84,
 'N': 85,
 'O': 86,
 'P': 87,
 'Q': 88,
 '

In [9]:
model = models.WordLevel(vocab=vocab, unk_token=SPECIALS["unk_token"])
tokenizer = Tokenizer(model)
pre_tokenizer = pre_tokenizers.Sequence(
    [
        pre_tokenizers.Split(Regex("\n"), behavior="isolated"),
        pre_tokenizers.Split(Regex("."), behavior="isolated"),
    ]
)
post_processor = processors.BertProcessing(
    sep=(SPECIALS["sep_token"], SPECIALS_IDS["sep_token"]),
    cls=(SPECIALS["cls_token"], SPECIALS_IDS["cls_token"]),
)
tokenizer.pre_tokenizer = pre_tokenizer
tokenizer.post_processor = post_processor

In [10]:
encoding = tokenizer.encode(text)
encoding

Encoding(num_tokens=784662, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [12]:
len(encoding.ids) == len(correct_ids) + 2, encoding.ids[1:-1] == correct_ids

(True, True)

In [14]:
encoding.ids[0], encoding.ids[-1]

(5, 6)

In [53]:
first_failure = None
for j, (i, c, t, s) in enumerate(zip(encoding.ids, correct_ids, encoding.tokens, text, strict=False)):
    if i != c and first_failure is None:
        print(f"{i}, {c}, {t}, {s}")
        break
j

784659

In [48]:
unk_idx = [i for i, val in enumerate(encoding.ids) if val == 1]
print(unk_idx)
idx = [unk_idx[0] - 1, unk_idx[0], unk_idx[0] + 1]
print("encoding.ids:", [encoding.ids[i] for i in idx])
print("encoding.tokens:", [encoding.tokens[i] for i in idx])
print("string:", [text[i] for i in idx])
print("byte:", [byte[i] for i in idx])

[184356, 210723]
encoding.ids: [18, 1, 213]
encoding.tokens: ['\x0b', '<unk>', 'Î']
string: ['\x0b', '\n', '\n']
byte: [11, 10, 10]


In [49]:
pre = pre_tokenizer.pre_tokenize_str(text)
pre[184355], pre[184356], pre[184357]

(('\x0b', (184355, 184356)),
 ('\n\n', (184356, 184358)),
 ('Î', (184358, 184359)))

In [51]:
for i in range(len(SPECIALS)):
	print(i, i in encoding.ids)

0 False
1 True
2 False
3 False
4 False
5 False
6 False


[117, 13491, 13932, 16934, 19954, 21594, 50822, 57572, 61144, 61227]

encoding.ids: [53, 1, 43, 7]
encoding.tokens: ['.', '<unk>', '$', '\x00']
string: ['.', '\n', '\n', '$']
byte: [13, 10, 36, 55]


In [54]:
model = models.WordLevel(vocab=vocab, unk_token=SPECIALS["unk_token"])
tokenizer = Tokenizer(model)
pre_tokenizer = pre_tokenizers.Sequence(
	[
		pre_tokenizers.Split(Regex("\n"), behavior="isolated"),
		pre_tokenizers.Split(Regex("."), behavior="isolated"),
	]
)
tokenizer.pre_tokenizer = pre_tokenizer
encoding = tokenizer.encode(string)
encoding

Encoding(num_tokens=65534, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [55]:
for i in range(len(SPECIALS)):
	print(i, i in encoding.ids)

0 False
1 False
2 False
3 False
4 False
5 False
6 False


In [56]:
idx = [116, 117, 118, 119]
print("encoding.ids:", [encoding.ids[i] for i in idx])
print("encoding.tokens:", [encoding.tokens[i] for i in idx])
print("string:", [string[i] for i in idx])
print("byte:", [byte[i] for i in idx])

encoding.ids: [53, 17, 17, 43]
encoding.tokens: ['.', '\n', '\n', '$']
string: ['.', '\n', '\n', '$']
byte: [13, 10, 36, 55]


In [57]:
test_encoding(encoding)

ValueError: byte:---80---, id:---151---, token:------, string:------